# Reusable Template — Retirement Plan Model

Swap in your own numbers to analyze a **different** retirement plan (a
different saver, a different goal, a different ladder) with the same NumPy
techniques from Chapter 3.

**To reuse this notebook for a new project:**
1. Edit only the `# >>> EDIT HERE <<<` cell in each of Sections 1, 2, and 4 —
   everything else is generic and runs unchanged.
2. Keep the bond ladder's `coupons`/`maturities` and `liabilities` the same
   length (an *n*-year ladder needs *n* bonds and *n* liability values).
3. Re-run the whole notebook (Kernel → Restart & Run All) after editing.

Read **Section 5 (Limitations)** before you trust or act on any output from
this template — the checks in the notebook are necessary but not sufficient
for the model to be a good fit for a real decision.


In [ ]:
import numpy as np
from numpy.linalg import solve

np.set_printoptions(precision=2, suppress=True)


## 1. Accumulation phase — edit this cell for your project

In [ ]:
# >>> EDIT HERE <<<
initial_balance = 20000
monthly_contribution = 500
annual_return = 0.07
years_to_retirement = 30
# >>> END EDIT <<<

monthly_rate = (1 + annual_return)**(1/12) - 1
months = np.arange(0, years_to_retirement * 12 + 1)
growth_factors = (1 + monthly_rate) ** months

balance = (initial_balance * growth_factors
           + monthly_contribution * ((growth_factors - 1) / monthly_rate))

print(f"Balance at retirement (month {months[-1]}, year {years_to_retirement}): "
      f"${balance[-1]:,.2f}")


## 2. Retirement income bond ladder — edit this cell for your project

`coupons[j]` and `maturities[j]` describe bond `j`. `liabilities[i]` is the
withdrawal need in year `i+1`. All three arrays must have matching length —
an *n*-bond ladder funds exactly *n* years of withdrawals.

In [ ]:
# >>> EDIT HERE <<<
coupons = np.array([0.03, 0.035, 0.04])   # annual coupon rate per bond
maturities = np.array([1, 2, 3])          # maturity year per bond
liabilities = np.array([40000, 42000, 44000])  # withdrawal need per year
# >>> END EDIT <<<

n = len(coupons)
assert len(maturities) == n and len(liabilities) == n, \
    "coupons, maturities, and liabilities must all be the same length"

A = np.zeros((n, n))
for i in range(1, n + 1):
    for j in range(n):
        if i < maturities[j]:
            A[i - 1, j] = coupons[j]
        elif i == maturities[j]:
            A[i - 1, j] = coupons[j] + 1

b = liabilities


## 3. Validity checks

Catch the most common reasons a cash-flow-matching problem is not solvable
*at all* — these do not tell you whether the plan is financially sound (see
Section 5), only whether the math is solvable.

In [ ]:
from numpy.linalg import matrix_rank

rank = matrix_rank(A)
if rank < n:
    raise ValueError(
        "The cash-flow matrix A is singular — cannot solve. This usually "
        "means two bonds have the same maturity year, or a maturity year "
        "in `maturities` doesn't fall within 1..n."
    )

if np.any(liabilities <= 0):
    print("WARNING: a liability is <= 0 — check your `liabilities` array.")

print("Validity checks passed. Cash-flow matrix A:\n", A)


## 4. Solve the ladder, then run the Monte Carlo sustainability check

In [ ]:
x = solve(A, b)
print("Face value to buy of each bond:", np.round(x, 2))
print("Check A @ x == b:", np.round(A @ x, 2))
print("Total dollars needed today for the ladder: $%.2f" % np.sum(x))


In [ ]:
def simulate_withdrawals(start_balance, annual_withdrawal, mu, sigma,
                          n_years=30, n_scenarios=1000, seed=None):
    """Simulate `n_scenarios` possible futures of `n_years` each, where the
    portfolio earns a random annual return ~ N(mu, sigma) and then
    `annual_withdrawal` is subtracted. Returns (ending_balance, depletion_year).

    depletion_year[k] == -1 means scenario k never ran out of money.
    """
    if seed is not None:
        np.random.seed(seed)
    bal = np.full(n_scenarios, float(start_balance))
    depletion_year = np.full(n_scenarios, -1)

    for year in range(1, n_years + 1):
        returns = np.random.normal(mu, sigma, n_scenarios)
        bal = bal * (1 + returns) - annual_withdrawal
        newly_ruined = (bal <= 0) & (depletion_year == -1)
        depletion_year[newly_ruined] = year
        bal = np.maximum(bal, 0)

    return bal, depletion_year


# >>> EDIT HERE <<<
annual_withdrawal = 40000
mu, sigma = 0.06, 0.12
n_years = 30
n_scenarios = 1000
random_seed = 1
# >>> END EDIT <<<

start_balance = balance[-1] - np.sum(x)  # remaining portfolio after buying the ladder
ending_balance, depletion_year = simulate_withdrawals(
    start_balance, annual_withdrawal, mu, sigma, n_years, n_scenarios, random_seed
)

prob_ruin = np.mean(depletion_year != -1)
print(f"Starting balance after funding the ladder: ${start_balance:,.2f}")
print(f"Probability of depleting savings within {n_years} years: {prob_ruin*100:.1f}%")
print(f"Median ending balance : ${np.median(ending_balance):,.2f}")
print(f"Mean ending balance   : ${np.mean(ending_balance):,.2f}")
print(f"10th percentile       : ${np.percentile(ending_balance, 10):,.2f}")
print(f"90th percentile       : ${np.percentile(ending_balance, 90):,.2f}")

ruined = depletion_year[depletion_year != -1]
if len(ruined):
    print(f"Average depletion year (among ruined scenarios): {np.mean(ruined):.1f}")


## 5. Limitations — read before reusing on a new plan

Every part of this template is a **simplified, assumption-driven** model.
That single sentence is the source of every limitation below.

### ✅ Reasonable to use when
- You want a **rough, order-of-magnitude** check on whether a savings plan
  is broadly on track, or to compare *relative* effects of changing one
  input (contribution, retirement age, withdrawal rate).
- You're using the bond ladder (Sections 2–4, first half) for its intended,
  narrow purpose: locking in **known, near-term, fixed** withdrawal needs
  with **investment-grade bonds you'd actually hold to maturity** — this
  piece is genuinely low-risk math, not a statistical estimate.
- You clearly label the Monte Carlo probability of ruin as *what this
  specific set of assumptions produces*, not a validated, historically
  back-tested probability.
- The time horizon and dollar amounts are being used for **planning and
  intuition-building**, with a professional or more detailed tool used
  before any irreversible action.

### 🚫 Not recommended when
- **Treating `annual_return`, `mu`, and `sigma` as known facts about the
  future.** They are typed-in assumptions. Small changes to `sigma` in
  particular can swing `prob_ruin` by a lot — see the reflection question
  in the practice notebook.
- **Ignoring taxes, inflation, Social Security/pension income, healthcare
  costs, required minimum distributions, or life expectancy uncertainty.**
  None of these are modeled here.
- **Assuming annual returns are independent and normally distributed.**
  Real markets have fatter tails (larger, more frequent extreme moves) and
  returns are often correlated year to year — this model's Monte Carlo
  step will generally **understate** real tail risk.
- **A fixed, never-adjusted annual withdrawal.** Real financial plans often
  use dynamic withdrawal rules (spend less after a bad year) — a fixed
  withdrawal, as modeled here, is the more pessimistic, less realistic case
  in bad-return scenarios and the more optimistic, unrealistic case in very
  good ones.
- **Bond default/credit risk is not modeled** in the ladder — the ladder
  assumes every coupon and every face value is paid in full, on time. Using
  it with anything other than very high-quality bonds understates risk.
- **Any real, irreversible retirement decision** — retiring, annuitizing,
  rolling over a pension, or changing a withdrawal strategy. Use this to
  build intuition and ask better questions of a financial professional, not
  as the final analysis.
